# 02 — Run QA and verify taxonomy reports

Runs SA metric branches on Testable QA, exports taxonomy HTML, and verifies
that each branch's **target classification** differentiates correctly
(Bug → FAIL on target; BugFX/TCC/CC → not FAIL on target).

Copy `.env.example` to `.env.local` and fill in credentials before running.

## Parameters

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "lib" / "sa_qa.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib.sa_metrics import (
    SA_TESTING_TYPE,
    branches_for_metrics,
    report_output_dir,
)
from lib.sa_qa import run_taxonomy_batch, verify_taxonomy_dir

VERSION = "2.6"
METRICS = "all"  # or "DOV" or "EPI,DOV"
ENV_FILE = str(ROOT / ".env.local")
DRY_RUN = False
REFRESH_BRANCHES = True
SKIP_VERIFY = False
KEEP_INTERMEDIATE = False  # False = HTML-only after export

branch_list = branches_for_metrics(METRICS, VERSION)
branches_csv = ",".join(branch_list)
print(f"Will run {len(branch_list)} branches")
for i, name in enumerate(branch_list, 1):
    print(f"  {i:2d}. {name}")

## 1. Run taxonomy batch on QA

In [ ]:
rc, batch_dir = run_taxonomy_batch(
    env_file=ENV_FILE,
    branches_csv=branches_csv,
    dry_run=DRY_RUN,
    refresh_branches=REFRESH_BRANCHES,
    export_html=True,
    html_only=not KEEP_INTERMEDIATE,
)
if rc != 0:
    raise RuntimeError(f"Batch run failed with exit code {rc}")

## 2. Verify HTML reports

In [ ]:
print("Batch directory:", batch_dir)

if SKIP_VERIFY:
    print("Skipped verification (SKIP_VERIFY=True)")
else:
    failed = verify_taxonomy_dir(str(batch_dir))
    if failed:
        raise RuntimeError(f"{failed} report(s) failed verification")
    print("All reports passed verification.")